# Kahnn `nano` — Colab GPU (T4/L4)

Train **nano** (~1.8M, ~37M tok) with `train_universal.py` on free Colab GPU.

1. Runtime → Change runtime type → **GPU**
2. Run cells in order; mount Drive for ckpts (~846 MiB each)

CPU box ref (2026-09-08): ~**1.07–1.23k tok/s**. T4 should be **much faster** — **measure `tps=` on smoke**; no invented GPU bench. See `docs/COLAB_GPU.md`.


## 0 — GPU


In [ ]:
!nvidia-smi
import torch
print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)


## 1 — Deps (keep Colab torch CUDA; do not pip install CPU torch)


In [ ]:
%pip install -q tiktoken tqdm numpy
%pip install -q bitsandbytes || echo "bitsandbytes optional — skip OK"
import tiktoken, tqdm, numpy, torch
print("ok", torch.__version__, torch.cuda.is_available())


## 2 — Clone + Drive


In [ ]:
from pathlib import Path
from google.colab import drive
DRIVE=Path("/content/drive/MyDrive/kahnn"); REPO=Path("/content/kahnn")
BRANCH="main"  # or feat/colab-nano-gpu before merge
USE_DRIVE=True
if USE_DRIVE:
    drive.mount("/content/drive", force_remount=False)
    (DRIVE/"runs/nano_colab").mkdir(parents=True, exist_ok=True)
    (DRIVE/"data").mkdir(parents=True, exist_ok=True)
if not REPO.exists():
    !git clone --branch {BRANCH} --depth 1 https://github.com/AFKmoney/kahnn.git {REPO}
%cd /content/kahnn
!git rev-parse --short HEAD


## 3 — Corpus (`data/DATA.md`) — Drive upload, Colab upload, or minimal rebuild


In [ ]:
from pathlib import Path
import shutil
DATA=Path("/content/kahnn/data"); DATA.mkdir(parents=True, exist_ok=True)
CORPUS=DATA/"corpus.txt"
src=Path("/content/drive/MyDrive/kahnn/data/corpus.txt")
if not CORPUS.exists() and src.exists():
    shutil.copy2(src, CORPUS)
if not CORPUS.exists():
    try:
        from google.colab import files
        up=files.upload()
        for n,b in up.items():
            CORPUS.write_bytes(b); break
    except Exception as e:
        print("upload skipped", e)
print("corpus", CORPUS.exists(), CORPUS.stat().st_size if CORPUS.exists() else None)


### Rebuild minimal if needed


In [ ]:
from pathlib import Path
import urllib.request, time
CORPUS=Path("/content/kahnn/data/corpus.txt"); RAW=Path("/content/kahnn/data/raw"); RAW.mkdir(parents=True, exist_ok=True)
IDS=[("1342","pride.txt"),("11","alice.txt"),("84","frank.txt"),("1661","sherlock.txt"),("2701","moby.txt")]
def fetch(gid,name):
    out=RAW/name
    if out.exists() and out.stat().st_size>10000: return out
    err=None
    for url in [f"https://www.gutenberg.org/files/{gid}/{gid}-0.txt", f"https://www.gutenberg.org/ebooks/{gid}.txt.utf-8"]:
        try:
            print("GET",url); urllib.request.urlretrieve(url,out)
            if out.stat().st_size>10000: return out
        except Exception as e:
            err=e; time.sleep(0.5)
    raise RuntimeError(err)
if CORPUS.exists() and CORPUS.stat().st_size>1_000_000:
    print("skip", CORPUS.stat().st_size)
else:
    parts=[fetch(g,n).read_text(encoding="utf-8",errors="ignore") for g,n in IDS]
    code=[]
    root=Path("/content/kahnn")
    for p in root.rglob("*"):
        if p.is_file() and p.suffix in {".py",".sh",".md"} and not any(x in p.parts for x in ("data","runs",".git","__pycache__","notebooks")):
            code.append(p.read_text(encoding="utf-8",errors="ignore"))
    base="\n\n".join(parts+["\n".join(code)])
    CORPUS.write_text(base*3, encoding="utf-8"); print("wrote", CORPUS.stat().st_size)
    d=Path("/content/drive/MyDrive/kahnn/data/corpus.txt")
    if d.parent.exists(): d.write_bytes(CORPUS.read_bytes())


## 4 — Optional resume from Drive / CPU box (`ckpt_1500+`)


In [ ]:
from pathlib import Path
import shutil
OUT=Path("/content/kahnn/runs/nano_colab"); OUT.mkdir(parents=True, exist_ok=True)
RESUME=None  # e.g. "/content/drive/MyDrive/kahnn/runs/from_cpu/ckpt_1500.pt"
drv=Path("/content/drive/MyDrive/kahnn/runs/nano_colab")
if drv.exists():
    for p in drv.glob("ckpt_*.pt"):
        if not (OUT/p.name).exists(): shutil.copy2(p, OUT/p.name)
if RESUME:
    rp=Path(RESUME); local=OUT/rp.name
    if rp.resolve()!=local.resolve(): shutil.copy2(rp, local); RESUME=str(local)
    print("resume", RESUME, Path(RESUME).stat().st_size)
else:
    print("cold start")
print(sorted(p.name for p in OUT.glob("ckpt_*.pt")))


## 5 — Smoke (first loss/tps) — measure here vs ~1.2k CPU


In [ ]:
%cd /content/kahnn
!nvidia-smi -L
!python train_universal.py --data /content/kahnn/data/corpus.txt --output /content/kahnn/runs/nano_colab --config nano --device cuda --micro-batch 8 --seq-len 256 --grad-accum 2 --log-every 1 --checkpoint-every 500 --smoke-steps 20


## 6 — Full nano train (or resume). OOM → micro-batch 4 + `--mod` + `--activation-checkpointing`


In [ ]:
%cd /content/kahnn
from pathlib import Path
OUT="/content/kahnn/runs/nano_colab"
resume=""
try: RESUME
except NameError: RESUME=None
if RESUME: resume=f"--resume {RESUME}"
else:
    numbered=[]
    for p in Path(OUT).glob("ckpt_*.pt"):
        if p.stem.startswith("ckpt_") and p.stem[5:].isdigit(): numbered.append((int(p.stem[5:]), p))
    if numbered: resume=f"--resume {sorted(numbered)[-1][1]}"; print(resume)
cmd=f"python train_universal.py --data /content/kahnn/data/corpus.txt --output {OUT} --config nano --device cuda --micro-batch 8 --seq-len 256 --grad-accum 2 --log-every 10 --checkpoint-every 500 {resume}"
print(cmd); get_ipython().system(cmd)


## 7 — Save ckpts to Drive


In [ ]:
from pathlib import Path
import shutil
src=Path("/content/kahnn/runs/nano_colab"); dst=Path("/content/drive/MyDrive/kahnn/runs/nano_colab"); dst.mkdir(parents=True, exist_ok=True)
for p in sorted(src.glob("ckpt_*.pt")):
    shutil.copy2(p, dst/p.name); print("copied", p.name, p.stat().st_size)
if (src/"train_universal.log").exists(): shutil.copy2(src/"train_universal.log", dst/"train_universal.log")
!ls -lh /content/drive/MyDrive/kahnn/runs/nano_colab | tail


## 8 — teach.py after pretrain


In [ ]:
%cd /content/kahnn
from pathlib import Path
CKPT="/content/kahnn/runs/nano_colab/ckpt_final.pt"
if not Path(CKPT).exists():
    numbered=[(int(p.stem[5:]),p) for p in Path("/content/kahnn/runs/nano_colab").glob("ckpt_*.pt") if p.stem.startswith("ckpt_") and p.stem[5:].isdigit()]
    if numbered: CKPT=str(sorted(numbered)[-1][1])
print("Using", CKPT)
!python teach.py teach --text "La capitale du Canada est Ottawa." --resume {CKPT} --config nano --device cuda --output /content/kahnn/runs/teach_colab
!python teach.py probe --text "La capitale du Canada est" --resume /content/kahnn/runs/teach_colab/ckpt_teach.pt --config nano --device cuda


## Notes
Free Colab disconnects — use Drive. Pro may get L4. Never invent GPU tps; read smoke logs.
